In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import random
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, losses, InputExample
from sentence_transformers.evaluation import TripletEvaluator
from torch.utils.data import DataLoader

model_name = "sentence-transformers/all-mpnet-base-v2"

labels = ['amusement', 'anger', 'awe', 'contentment',
          'disgust', 'excitement', 'fear', 'sadness']
label2id = {l: i for i, l in enumerate(labels)}

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/datasets/ul1oinasrchr/go-emotions/goemotions_balanced.csv"
)["train"]

splits = dataset.train_test_split(test_size=0.2, seed=42)
train = splits["train"]
tmp = splits["test"].train_test_split(test_size=0.5, seed=42)
val = tmp["train"]
test = tmp["test"]

model = SentenceTransformer(model_name, device="cuda")

train_examples = [
    InputExample(texts=[row["text"]], label=label2id[row["label"]])
    for row in train
]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

train_loss = losses.BatchSemiHardTripletLoss(model)

def make_triplets(ds, n=2000, seed=42):
    random.seed(seed)
    by_label = {}
    for row in ds:
        by_label.setdefault(row["label"], []).append(row["text"])
    labs = [l for l in by_label if len(by_label[l]) >= 2]
    a, p, neg = [], [], []
    for _ in range(n):
        lab = random.choice(labs)
        x, y = random.sample(by_label[lab], 2)
        nlab = random.choice([l for l in labs if l != lab])
        a.append(x); p.append(y); neg.append(random.choice(by_label[nlab]))
    return a, p, neg

a, p, neg = make_triplets(val)
evaluator = TripletEvaluator(anchors=a, positives=p, negatives=neg, name="emotion-val")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=4,
    warmup_steps=100,
    evaluation_steps=500,
    output_path="/kaggle/working/emotion-embedder",
    save_best_model=True,
    show_progress_bar=True,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss
